# BulkFormer-Atlas: sample embeddings

This notebook follows the original BulkFormer workflow: load the model, prepare
expression data, extract features and save the result. Atlas returns a **1,024-dimensional
CLS embedding** for each sample. The 512-dimensional contrastive projection is not
the representation used in our PRIMO submission.

Run the installation commands in the repository README first. Select your GPU
before starting Jupyter, for example with `CUDA_VISIBLE_DEVICES=0 jupyter lab`.
The outputs below were generated on a CUDA GPU using eight **synthetic** RNA-seq
count profiles. They illustrate the API; they are not a biological evaluation.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch

if Path.cwd().name == "notebooks":
    os.chdir("..")

device = "cuda"
assert torch.cuda.is_available(), "This tutorial uses a CUDA GPU."
torch.set_num_threads(4)
print(torch.cuda.get_device_name(0))


NVIDIA A800 80GB PCIe


### Load the released checkpoint

Download the full weight file and its companion resources into `model/atlas/`
before running the notebook. You can use the Google Drive folder linked in the
README or uncomment the download line below. The loader checks file integrity.


In [2]:
from bulkformer import download_model, load_atlas
from bulkformer import prepare_atlas_input, select_variable_genes, extract_atlas

model_dir = Path("model/atlas")
# download_model("atlas", model_dir)  # Run once if resources are not downloaded.
model, gene_info, normalization_info = load_atlas(model_dir, device=device)
print(model.release_metadata["checkpoint_id"])
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
print(f"{len(gene_info):,} coding genes; {len(normalization_info):,} normalization genes")


atlas-infonce-epoch10-update320000
242,372,356 parameters
22,636 coding genes; 41,623 normalization genes


### Prepare the expression matrix

Rows are samples and columns are human Ensembl gene IDs. `input_scale="counts"`
converts counts to TPM before taking **log2(TPM + 1)**. Normalization uses matched
genes on the released 41,623-gene axis, then retains the 22,636 coding genes.
Provide the full available count matrix before selecting an input panel.

For an already normalized matrix, use `"tpm"`, `"log1p_tpm"` (natural log), or
`"log2_tpm"` as appropriate. This avoids repeating library normalization.


In [3]:
count_df = pd.read_csv("examples/demo_counts.csv", index_col=0)
print("Count matrix:", count_df.shape)
count_df.iloc[:5, :6]


Count matrix: (8, 41671)


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938
demo_01,123,330,66,111,565,185
demo_02,185,131,365,68,304,197
demo_03,153,101,357,147,246,161
demo_04,112,86,152,91,92,58
demo_05,151,339,163,96,376,234


In [4]:
expression, coverage = prepare_atlas_input(
    count_df, gene_info, normalization_info, input_scale="counts"
)
print("Aligned coding-gene matrix:", expression.shape)
coverage


Aligned coding-gene matrix: (8, 22636)


,observed_genes,missing_fraction
demo_01,22636,0.0
demo_02,22636,0.0
demo_03,22636,0.0
demo_04,22636,0.0
demo_05,22636,0.0
demo_06,22636,0.0
demo_07,22636,0.0
demo_08,22636,0.0


### Select one fixed gene panel

Our PRIMO workflow used the 4,096 genes with the highest expression variance
within each unlabeled dataset. The same panel is used for every sample in that
dataset. For prospective prediction, select the panel using **training samples
only** and reuse the saved Ensembl IDs for validation and test samples. Panel genes
must be observed in every input sample; measured zero expression is allowed.

The demo uses all eight synthetic profiles to illustrate whole-cohort selection.
This 4,096-gene panel is separate from the official PRIMO **+ HVG** baseline, which
concatenates 1,200 expression features with the model embedding.


In [5]:
gene_indices = select_variable_genes(expression, n_genes=4096)
selected_genes = gene_info.iloc[gene_indices][["ensembl_gene", "symbol"]]
print("Selected genes:", len(selected_genes))
selected_genes.head(8)


Selected genes: 4096


,ensembl_gene,symbol
1,ENSG00000000005,TNMD
2,ENSG00000000419,DPM1
4,ENSG00000000460,C1ORF112
13,ENSG00000001561,ENPP4
17,ENSG00000001630,CYP51A1
23,ENSG00000002587,HS3ST1
24,ENSG00000002726,AOC1
31,ENSG00000003056,M6PR


### Extract sample-level features

Each sample is encoded with a CLS token and the fixed panel. Inference uses no
token masking, noise or dropout. BF16 is used on a compatible GPU; pass
`precision="fp32"` for FP32 inference.


In [6]:
sample_embeddings = extract_atlas(
    model, expression, gene_indices, batch_size=2, progress=False
)
print("Sample embeddings:", sample_embeddings.shape)
print("All values finite:", np.isfinite(sample_embeddings).all())
pd.DataFrame(sample_embeddings[:, :6], index=count_df.index,
             columns=[f"feature_{i}" for i in range(6)])


Sample embeddings: (8, 1024)
All values finite: True


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5
demo_01,0.145634,0.084510,0.133271,-0.065781,-0.027089,0.063000
demo_02,0.136776,0.029496,0.154638,-0.098114,-0.023243,0.025998
demo_03,0.101951,0.087208,0.114090,-0.090905,-0.083793,0.023064
demo_04,0.123683,0.051533,0.130249,-0.070658,-0.088433,0.034576
demo_05,0.117645,0.091489,0.172871,-0.111936,-0.053397,0.067737
demo_06,0.111208,0.065128,0.144108,-0.083352,-0.043953,0.089068
demo_07,0.112892,0.080347,0.140622,-0.104623,-0.007350,0.045418
demo_08,0.100023,0.091080,0.150523,-0.100359,-0.045905,0.008697


In [7]:
pd.DataFrame({
    "observed_genes": coverage["observed_genes"],
    "embedding_l2_norm": np.linalg.norm(sample_embeddings, axis=1)
})


,observed_genes,embedding_l2_norm
demo_01,22636,6.980385
demo_02,22636,6.974100
demo_03,22636,6.942863
demo_04,22636,6.971153
demo_05,22636,6.967154
demo_06,22636,6.958422
demo_07,22636,6.978196
demo_08,22636,6.975574


### Save embeddings and the input panel

Keep the sample IDs, selected genes and checkpoint identity with the feature
matrix. Fit downstream preprocessing and classifiers on training folds only.


In [8]:
output_dir = Path("outputs/atlas_demo")
output_dir.mkdir(parents=True, exist_ok=True)
selected_genes.to_csv(output_dir / "input_gene_panel.csv", index=False)
np.savez_compressed(
    output_dir / "sample_embeddings.npz",
    embeddings=sample_embeddings,
    sample_ids=count_df.index.to_numpy(dtype=str),
    gene_ids=selected_genes["ensembl_gene"].to_numpy(dtype=str),
    checkpoint_id=model.release_metadata["checkpoint_id"],
)
print(output_dir / "sample_embeddings.npz")


outputs/atlas_demo/sample_embeddings.npz


For the architecture and official leaderboard results, see
the [Atlas model card](../docs/models/atlas.md) and [PRIMO protocol](../benchmarks/primo/README.md).
